In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from huggingface_hub import login
from typing import Dict
import os
import torch



# 1️⃣ Load & preprocess dataset
ds = load_dataset("bond0213/Python-Questions-from-Stack-Overflow-dataset", split="train")

def format_example(ex: Dict[str, str]) -> Dict[str, str]:
    text = f"{ex['instruction']}\n\n{ex['input']}\n\n{ex['output']}"
    return {"text": text}

ds = ds.map(format_example)

# 2️⃣ Hugging Face login (token from environment variable)
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN environment variable not set.")

login(token=hf_token)

# 3️⃣ Load tokenizer and prepare dataset
tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", use_fast=True)
tok.pad_token = tok.eos_token

def tokenize_example(examples: Dict[str, str]) -> Dict[str, list]:
    return tok(examples["text"], truncation=True, max_length=512)

ds = ds.map(tokenize_example, batched=True, remove_columns=["instruction", "input", "output", "text"])  # Clean up

# 4️⃣ Load 4-bit quantized base model with CPU offloading
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_compute_dtype="float16",
    bnb_4bit_compute_dtype=torch.float16,     
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=quantization_config,
    device_map="auto"
)

# 5️⃣ Configure LoRA
lora_cfg = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# 6️⃣ Wrap the base model with PEFT (LoRA)
model = get_peft_model(base_model, lora_cfg)

# Ensure the PEFT model has a valid .device before training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # Explicitly move the model to the correct device

# 7️⃣ Setup SFT Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="mistral-python-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    num_train_epochs=1,
    lr_scheduler_type="constant",
)

# The `SFTTrainer` takes only model, dataset, args, peft_config and other packing options
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=training_args,
    peft_config=lora_cfg,
)

# 8️⃣ Train model
trainer.train()

# 9️⃣ Save final model
model.save_pretrained("mistral-python-lora")

print("Training complete. Model saved to ./mistral-python-lora")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
